# Huggingface EmbeddingsHuggingFace Transformers Embeddings GuideThis module shows how to generate embeddings using HuggingFace Transformers library.Covers BERT, RoBERTa, DistilBERT, and other transformer models.Install: pip install transformers torch sentence-transformers

## SetupImport required libraries:

In [ ]:
import torchfrom transformers import AutoTokenizer, AutoModelimport numpy as npfrom scipy.spatial.distance import cosine

## Example 1 Basic Bert EmbeddingsGenerate embeddings using BERT

In [ ]:
def example_1_basic_bert_embeddings():    """Generate embeddings using BERT"""    print("\n" + "="*70)    print("Example 1: Basic BERT Embeddings")    print("="*70)        # Load pre-trained BERT model and tokenizer    model_name = "bert-base-uncased"    tokenizer = AutoTokenizer.from_pretrained(model_name)    model = AutoModel.from_pretrained(model_name)        # Put model in evaluation mode    model.eval()        # Example texts    texts = [        "Machine learning is fascinating",        "AI and machine learning are related",        "I love pizza and pasta"    ]        print("\nGenerating BERT embeddings...")    print(f"Model: {model_name}")    print(f"Hidden size: {model.config.hidden_size}")        embeddings = []        for text in texts:        # Tokenize        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)                # Generate embeddings        with torch.no_grad():            outputs = model(**inputs)                # Get [CLS] token embedding (first token)        # This represents the entire sentence        cls_embedding = outputs.last_hidden_state[:, 0, :].numpy()        embeddings.append(cls_embedding[0])                print(f"\nText: '{text}'")        print(f"  Shape: {cls_embedding.shape}")        print(f"  First 5 values: {cls_embedding[0][:5]}")        # Calculate similarities    print("\n" + "-"*70)    print("Similarity Scores:")    print("-"*70)        for i in range(len(texts)):        for j in range(i+1, len(texts)):            similarity = 1 - cosine(embeddings[i], embeddings[j])            print(f"\n'{texts[i]}'")            print(f"  vs")            print(f"'{texts[j]}'")            print(f"  Similarity: {similarity:.4f}")

## Example 2 Mean PoolingUse mean pooling instead of CLS token

In [ ]:
def example_2_mean_pooling():    """Use mean pooling instead of CLS token"""    print("\n" + "="*70)    print("Example 2: Mean Pooling for Better Sentence Embeddings")    print("="*70)        model_name = "bert-base-uncased"    tokenizer = AutoTokenizer.from_pretrained(model_name)    model = AutoModel.from_pretrained(model_name)    model.eval()        def mean_pooling(model_output, attention_mask):        """        Mean pooling - average all token embeddings        weighted by attention mask        """        token_embeddings = model_output[0]  # First element = token embeddings        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)        texts = [        "The cat sits on the mat",        "A feline rests on the carpet",        "Pizza is my favorite food"    ]        print("\nComparing CLS token vs Mean Pooling:")        for text in texts:        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)                with torch.no_grad():            outputs = model(**inputs)                # Method 1: CLS token        cls_embedding = outputs.last_hidden_state[:, 0, :].numpy()[0]                # Method 2: Mean pooling        mean_embedding = mean_pooling(outputs, inputs['attention_mask']).numpy()[0]                print(f"\nText: '{text}'")        print(f"  CLS embedding norm: {np.linalg.norm(cls_embedding):.4f}")        print(f"  Mean embedding norm: {np.linalg.norm(mean_embedding):.4f}")        print("\n💡 Mean pooling often works better for sentence similarity!")

## Example 3 Different ModelsCompare different transformer models

In [ ]:
def example_3_different_models():    """Compare different transformer models"""    print("\n" + "="*70)    print("Example 3: Comparing Different Transformer Models")    print("="*70)        models = {        "BERT Base": "bert-base-uncased",        "DistilBERT": "distilbert-base-uncased",        "RoBERTa": "roberta-base",    }        text = "Natural language processing with transformers"        print(f"\nText: '{text}'")    print("\nModel Comparison:")    print("-"*70)        for model_name, model_id in models.items():        tokenizer = AutoTokenizer.from_pretrained(model_id)        model = AutoModel.from_pretrained(model_id)        model.eval()                inputs = tokenizer(text, return_tensors="pt")                with torch.no_grad():            outputs = model(**inputs)                embedding = outputs.last_hidden_state[:, 0, :].numpy()[0]                print(f"\n{model_name}:")        print(f"  Model ID: {model_id}")        print(f"  Embedding dim: {len(embedding)}")        print(f"  Norm: {np.linalg.norm(embedding):.4f}")        print(f"  First 5 values: {embedding[:5]}")

## Example 4 Batch ProcessingProcess multiple texts efficiently in batches

In [ ]:
def example_4_batch_processing():    """Process multiple texts efficiently in batches"""    print("\n" + "="*70)    print("Example 4: Efficient Batch Processing")    print("="*70)        model_name = "bert-base-uncased"    tokenizer = AutoTokenizer.from_pretrained(model_name)    model = AutoModel.from_pretrained(model_name)    model.eval()        texts = [        "Machine learning is a subset of AI",        "Deep learning uses neural networks",        "Natural language processing handles text",        "Computer vision processes images",        "Robotics combines hardware and software"    ]        print(f"\nProcessing {len(texts)} texts in a batch...")        # Batch tokenization    inputs = tokenizer(        texts,        padding=True,        truncation=True,        return_tensors="pt",        max_length=128    )        print(f"Batch shape: {inputs['input_ids'].shape}")    print(f"  (batch_size={inputs['input_ids'].shape[0]}, seq_length={inputs['input_ids'].shape[1]})")        # Generate embeddings for all texts at once    with torch.no_grad():        outputs = model(**inputs)        # Extract CLS token embeddings    embeddings = outputs.last_hidden_state[:, 0, :].numpy()        print(f"\nEmbeddings shape: {embeddings.shape}")    print(f"  (num_texts={embeddings.shape[0]}, embedding_dim={embeddings.shape[1]})")        # Find most similar pair    print("\nFinding most similar pair:")    max_similarity = -1    max_pair = (0, 0)        for i in range(len(texts)):        for j in range(i+1, len(texts)):            similarity = 1 - cosine(embeddings[i], embeddings[j])            if similarity > max_similarity:                max_similarity = similarity                max_pair = (i, j)        print(f"\nMost similar texts:")    print(f"  '{texts[max_pair[0]]}'")    print(f"  '{texts[max_pair[1]]}'")    print(f"  Similarity: {max_similarity:.4f}")

## Example 5 Sentence TransformersUse sentence-transformers for optimized embeddings

In [ ]:
def example_5_sentence_transformers():    """Use sentence-transformers for optimized embeddings"""    print("\n" + "="*70)    print("Example 5: Sentence Transformers (Optimized)")    print("="*70)        from sentence_transformers import SentenceTransformer        # This model is specifically fine-tuned for sentence embeddings    model = SentenceTransformer('all-MiniLM-L6-v2')        texts = [        "A man is eating food",        "A person is consuming a meal",        "A dog is playing in the park",        "The weather is sunny today"    ]        print("\nUsing sentence-transformers library:")    print("Model: all-MiniLM-L6-v2 (optimized for speed & quality)")        # Generate embeddings (much simpler API!)    embeddings = model.encode(texts)        print(f"\nEmbeddings shape: {embeddings.shape}")    print(f"  Dimension: {embeddings.shape[1]}")        # Calculate similarity matrix    print("\nSimilarity Matrix:")    print("-"*70)        for i, text_i in enumerate(texts):        print(f"\n{i+1}. '{text_i}'")        for j, text_j in enumerate(texts):            if i != j:                similarity = 1 - cosine(embeddings[i], embeddings[j])                print(f"   vs {j+1}: {similarity:.4f}")

## Example 6 Multilingual EmbeddingsGenerate embeddings for multiple languages

In [ ]:
def example_6_multilingual_embeddings():    """Generate embeddings for multiple languages"""    print("\n" + "="*70)    print("Example 6: Multilingual Embeddings")    print("="*70)        from sentence_transformers import SentenceTransformer        # Multilingual model    model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')        texts = {        'English': 'Hello, how are you?',        'Spanish': '¡Hola, cómo estás?',        'French': 'Bonjour, comment allez-vous?',        'German': 'Hallo, wie geht es dir?',        'Chinese': '你好吗？',        'Japanese': 'こんにちは、お元気ですか？'    }        print("\nMultilingual Model: paraphrase-multilingual-MiniLM-L12-v2")    print("Supports 50+ languages")        # Generate embeddings    text_list = list(texts.values())    embeddings = model.encode(text_list)        print(f"\nEmbeddings shape: {embeddings.shape}")        # Compare similarities    print("\nCross-language Similarity Scores:")    print("-"*70)        languages = list(texts.keys())    english_idx = 0        for i, (lang, text) in enumerate(texts.items()):        if i != english_idx:            similarity = 1 - cosine(embeddings[english_idx], embeddings[i])            print(f"English <-> {lang:10s}: {similarity:.4f}")        print("\n💡 All mean 'Hello, how are you?' - high similarity!")

## Example 7 Custom Pooling StrategiesDifferent ways to pool token embeddings

In [ ]:
def example_7_custom_pooling_strategies():    """Different ways to pool token embeddings"""    print("\n" + "="*70)    print("Example 7: Custom Pooling Strategies")    print("="*70)        model_name = "bert-base-uncased"    tokenizer = AutoTokenizer.from_pretrained(model_name)    model = AutoModel.from_pretrained(model_name)    model.eval()        text = "Machine learning transforms data into insights"        inputs = tokenizer(text, return_tensors="pt")        with torch.no_grad():        outputs = model(**inputs)        token_embeddings = outputs.last_hidden_state[0]  # [seq_len, hidden_size]        # Different pooling strategies    strategies = {        "CLS Token": token_embeddings[0],        "Mean Pooling": torch.mean(token_embeddings, dim=0),        "Max Pooling": torch.max(token_embeddings, dim=0)[0],        "Last Token": token_embeddings[-1],    }        print(f"\nText: '{text}'")    print(f"Token embeddings shape: {token_embeddings.shape}")    print(f"  ({token_embeddings.shape[0]} tokens × {token_embeddings.shape[1]} dimensions)")        print("\nPooling Strategy Comparison:")    print("-"*70)        for strategy, embedding in strategies.items():        embedding_np = embedding.numpy()        print(f"\n{strategy}:")        print(f"  Shape: {embedding_np.shape}")        print(f"  Norm: {np.linalg.norm(embedding_np):.4f}")        print(f"  Mean: {embedding_np.mean():.4f}")        print(f"  Std: {embedding_np.std():.4f}")

## Example 8 Token Level EmbeddingsGet embeddings for individual tokens

In [ ]:
def example_8_token_level_embeddings():    """Get embeddings for individual tokens"""    print("\n" + "="*70)    print("Example 8: Token-Level Embeddings")    print("="*70)        model_name = "bert-base-uncased"    tokenizer = AutoTokenizer.from_pretrained(model_name)    model = AutoModel.from_pretrained(model_name)    model.eval()        text = "Natural language processing"        inputs = tokenizer(text, return_tensors="pt")    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])        with torch.no_grad():        outputs = model(**inputs)        token_embeddings = outputs.last_hidden_state[0]  # [seq_len, hidden_size]        print(f"\nText: '{text}'")    print(f"\nToken-by-Token Embeddings:")    print("-"*70)        for i, (token, embedding) in enumerate(zip(tokens, token_embeddings)):        embedding_np = embedding.numpy()        print(f"\nToken {i}: '{token}'")        print(f"  Embedding dim: {len(embedding_np)}")        print(f"  Norm: {np.linalg.norm(embedding_np):.4f}")        print(f"  First 5 values: {embedding_np[:5]}")

## Example 9 Comparing LayersCompare embeddings from different layers

In [ ]:
def example_9_comparing_layers():    """Compare embeddings from different layers"""    print("\n" + "="*70)    print("Example 9: Embeddings from Different Layers")    print("="*70)        model_name = "bert-base-uncased"    tokenizer = AutoTokenizer.from_pretrained(model_name)    model = AutoModel.from_pretrained(model_name, output_hidden_states=True)    model.eval()        text = "Transformers are powerful"        inputs = tokenizer(text, return_tensors="pt")        with torch.no_grad():        outputs = model(**inputs)        # Get embeddings from all layers    all_hidden_states = outputs.hidden_states  # Tuple of layer outputs        print(f"\nText: '{text}'")    print(f"Number of layers: {len(all_hidden_states)}")    print("  (Layer 0 = input embeddings, Layer 12 = final output)")        # Compare CLS token embedding across layers    print("\nCLS Token Embedding Across Layers:")    print("-"*70)        for i, hidden_state in enumerate(all_hidden_states):        cls_embedding = hidden_state[0, 0, :].numpy()  # CLS token        norm = np.linalg.norm(cls_embedding)        mean = cls_embedding.mean()                print(f"Layer {i:2d}: norm={norm:.4f}, mean={mean:.6f}")        print("\n💡 Higher layers capture more semantic meaning!")

## Example 10 Contextual EmbeddingsShow how context affects word embeddings

In [ ]:
def example_10_contextual_embeddings():    """Show how context affects word embeddings"""    print("\n" + "="*70)    print("Example 10: Contextual Embeddings")    print("="*70)        model_name = "bert-base-uncased"    tokenizer = AutoTokenizer.from_pretrained(model_name)    model = AutoModel.from_pretrained(model_name)    model.eval()        # Same word "bank" in different contexts    sentences = [        "I need to go to the bank to deposit money",        "The river bank was covered with flowers",        "The bank announced new interest rates"    ]        print("\nWord: 'bank' in different contexts")    print("-"*70)        bank_embeddings = []        for sentence in sentences:        inputs = tokenizer(sentence, return_tensors="pt")        tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])                with torch.no_grad():            outputs = model(**inputs)                token_embeddings = outputs.last_hidden_state[0]                # Find "bank" token        bank_idx = [i for i, token in enumerate(tokens) if 'bank' in token.lower()][0]        bank_embedding = token_embeddings[bank_idx].numpy()        bank_embeddings.append(bank_embedding)                print(f"\n'{sentence}'")        print(f"  'bank' at position {bank_idx}")        print(f"  Embedding norm: {np.linalg.norm(bank_embedding):.4f}")        # Compare similarity    print("\n" + "-"*70)    print("Contextual Similarity:")    print("-"*70)        for i in range(len(sentences)):        for j in range(i+1, len(sentences)):            similarity = 1 - cosine(bank_embeddings[i], bank_embeddings[j])            print(f"\nContext {i+1} vs Context {j+1}: {similarity:.4f}")        print("\n💡 Same word, different embeddings based on context!")

## MainRun all examples

In [ ]:
def main():    """Run all examples"""    print("\n" + "="*70)    print("HUGGINGFACE TRANSFORMERS EMBEDDINGS GUIDE")    print("="*70)        print("\nThis guide covers:")    print("  1. Basic BERT embeddings")    print("  2. Mean pooling vs CLS token")    print("  3. Different transformer models")    print("  4. Batch processing")    print("  5. Sentence transformers (optimized)")    print("  6. Multilingual embeddings")    print("  7. Custom pooling strategies")    print("  8. Token-level embeddings")    print("  9. Embeddings from different layers")    print("  10. Contextual embeddings")        try:        example_1_basic_bert_embeddings()        example_2_mean_pooling()        example_3_different_models()        example_4_batch_processing()        example_5_sentence_transformers()        example_6_multilingual_embeddings()        example_7_custom_pooling_strategies()        example_8_token_level_embeddings()        example_9_comparing_layers()        example_10_contextual_embeddings()                print("\n" + "="*70)        print("KEY TAKEAWAYS")        print("="*70)        print("""1. **HuggingFace Transformers**: Flexible but requires more code   - Use AutoModel and AutoTokenizer   - Choose pooling strategy (CLS, mean, max)   2. **Sentence Transformers**: Optimized for sentence embeddings   - Simpler API: model.encode(texts)   - Pre-trained for similarity tasks   3. **Pooling Matters**: Different strategies for different tasks   - CLS token: Traditional BERT approach   - Mean pooling: Often better for similarity   - Max pooling: Captures strongest features   4. **Context is Key**: Transformer embeddings are contextual   - Same word → different embeddings in different contexts   - This is the power of transformers!   5. **Choose the Right Model**:   - English only: BERT, RoBERTa, DistilBERT   - Multilingual: multilingual-BERT, XLM-RoBERTa   - Optimized: sentence-transformers models   6. **Batch Processing**: Always process multiple texts together   - Much faster than one-by-one   - GPU utilization improves        """)            except Exception as e:        print(f"\n❌ Error: {e}")        print("\nMake sure you have installed:")        print("  pip install transformers torch sentence-transformers")if __name__ == "__main__":    main()